# Graficas de pruebas de carga G1G2

Estas graficas resumen las pruebas de carga realizadas con G1 y G2 activos simultaneamente. Se analiza la latencia media, la latencia maxima y el porcentaje de mensajes que cumplen el requisito de latencia menor o igual a 200 ms para cargas baja, media y alta, usando envios de 1, 10 y 100 mensajes por cliente.

In [ ]:
from pathlib import Path
import csv
import html
import re
from statistics import mean

CARPETA = Path.cwd()
ORDEN_CARGA = {'baja': 0, 'media': 1, 'alta': 2}
ORDEN_GRUPO = {'G1': 0, 'G2': 1}
COLORES = {
    ('baja', 'G1'): '#2878b5',
    ('baja', 'G2'): '#6baed6',
    ('media', 'G1'): '#5aa469',
    ('media', 'G2'): '#98c379',
    ('alta', 'G1'): '#d95f02',
    ('alta', 'G2'): '#f4a261',
}


def parse_nombre(path):
    m = re.match(r'test_\d+_G1G2_(baja|media|alta)_(\d+)msg_(G[12])\.csv', path.name)
    if not m:
        raise ValueError(f'Nombre no reconocido: {path.name}')
    return m.group(1), int(m.group(2)), m.group(3)


def leer_csv(path):
    carga, mensajes, grupo = parse_nombre(path)
    latencias = []
    cumplen = 0

    with path.open(encoding='utf-8', newline='') as f:
        for fila in csv.DictReader(f):
            if fila.get('latency_ms') in (None, '', 'latency_ms'):
                continue
            try:
                latencia = float(fila['latency_ms'])
            except ValueError:
                continue
            latencias.append(latencia)
            if fila.get('meets_200ms') == 'yes':
                cumplen += 1

    total = len(latencias)
    return {
        'archivo': path.name,
        'carga': carga,
        'mensajes': mensajes,
        'grupo': grupo,
        'n_muestras': total,
        'latencia_media_ms': mean(latencias) if latencias else 0,
        'latencia_maxima_ms': max(latencias) if latencias else 0,
        'cumplimiento_200ms_pct': 100 * cumplen / total if total else 0,
    }


datos = sorted(
    [leer_csv(path) for path in CARPETA.glob('test_*_G1G2_*.csv')],
    key=lambda x: (x['mensajes'], ORDEN_CARGA[x['carga']], ORDEN_GRUPO[x['grupo']]),
)

for fila in datos:
    print(
        f"{fila['carga']:5} {fila['mensajes']:3} msg {fila['grupo']}: "
        f"media={fila['latencia_media_ms']:.2f} ms, "
        f"max={fila['latencia_maxima_ms']:.0f} ms, "
        f"cumple={fila['cumplimiento_200ms_pct']:.1f}%"
    )


def svg_barras_agrupadas(nombre, titulo, campo, etiqueta_y, maximo=None):
    mensajes = sorted({fila['mensajes'] for fila in datos})
    cargas = ['baja', 'media', 'alta']
    grupos = ['G1', 'G2']
    series = [(carga, grupo) for carga in cargas for grupo in grupos]
    valores = [fila[campo] for fila in datos]
    ymax = maximo if maximo is not None else max(valores) * 1.15
    ymax = ymax or 1

    ancho, alto = 1140, 540
    margen_izq, margen_der, margen_sup, margen_inf = 85, 210, 60, 95
    base = alto - margen_inf
    area_w = ancho - margen_izq - margen_der
    area_h = alto - margen_sup - margen_inf
    grupo_w = area_w / len(mensajes)
    barra = grupo_w / 8.2

    partes = [
        f"<svg xmlns='http://www.w3.org/2000/svg' width='{ancho}' height='{alto}' viewBox='0 0 {ancho} {alto}'>",
        "<rect width='100%' height='100%' fill='white'/>",
        f"<text x='{ancho/2}' y='30' text-anchor='middle' font-family='Arial' font-size='22' font-weight='700'>{html.escape(titulo)}</text>",
        f"<line x1='{margen_izq}' y1='{base}' x2='{ancho-margen_der}' y2='{base}' stroke='#333'/>",
        f"<line x1='{margen_izq}' y1='{margen_sup}' x2='{margen_izq}' y2='{base}' stroke='#333'/>",
        f"<text x='24' y='{margen_sup + area_h/2}' transform='rotate(-90 24 {margen_sup + area_h/2})' text-anchor='middle' font-family='Arial' font-size='14'>{html.escape(etiqueta_y)}</text>",
        f"<text x='{margen_izq + area_w/2}' y='{alto-18}' text-anchor='middle' font-family='Arial' font-size='14'>Numero de mensajes enviados</text>",
    ]

    for i in range(6):
        v = ymax * i / 5
        y = base - (v / ymax) * area_h
        partes.append(f"<line x1='{margen_izq}' y1='{y:.1f}' x2='{ancho-margen_der}' y2='{y:.1f}' stroke='#ddd'/>")
        partes.append(f"<text x='{margen_izq-10}' y='{y+4:.1f}' text-anchor='end' font-family='Arial' font-size='12'>{v:.1f}</text>")

    for i, msg in enumerate(mensajes):
        centro = margen_izq + i * grupo_w + grupo_w / 2
        partes.append(f"<text x='{centro:.1f}' y='{base+24}' text-anchor='middle' font-family='Arial' font-size='13'>{msg}</text>")
        inicio = centro - (len(series) * barra) / 2
        for j, (carga, grupo) in enumerate(series):
            fila = next(f for f in datos if f['mensajes'] == msg and f['carga'] == carga and f['grupo'] == grupo)
            valor = fila[campo]
            x = inicio + j * barra
            h = (valor / ymax) * area_h
            y = base - h
            partes.append(f"<rect x='{x:.1f}' y='{y:.1f}' width='{barra*0.82:.1f}' height='{h:.1f}' fill='{COLORES[(carga, grupo)]}'/>")
            partes.append(f"<text x='{x + barra*0.41:.1f}' y='{y-5:.1f}' text-anchor='middle' font-family='Arial' font-size='10'>{valor:.1f}</text>")

    leyenda_x = ancho - margen_der + 35
    for i, (carga, grupo) in enumerate(series):
        y = 52 + i * 23
        partes.append(f"<rect x='{leyenda_x}' y='{y}' width='16' height='16' fill='{COLORES[(carga, grupo)]}'/>")
        partes.append(f"<text x='{leyenda_x+22}' y='{y+13}' font-family='Arial' font-size='13'>{carga} {grupo}</text>")

    partes.append('</svg>')
    (CARPETA / nombre).write_text('\n'.join(partes), encoding='utf-8')


svg_barras_agrupadas('latencia_media_G1G2.svg', 'Latencia media en G1G2', 'latencia_media_ms', 'Latencia media (ms)')
svg_barras_agrupadas('latencia_maxima_G1G2.svg', 'Latencia maxima en G1G2', 'latencia_maxima_ms', 'Latencia maxima (ms)')
svg_barras_agrupadas('cumplimiento_200ms_G1G2.svg', 'Cumplimiento del umbral de 200 ms en G1G2', 'cumplimiento_200ms_pct', 'Cumplimiento (%)', maximo=100)

print('\nGraficas guardadas:')
for nombre in ['latencia_media_G1G2.svg', 'latencia_maxima_G1G2.svg', 'cumplimiento_200ms_G1G2.svg']:
    print(CARPETA / nombre)